<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Exercise: Add a Review Agent</h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

# Exercise: Add a Review Agent

**Purpose:**  
Improve the quality of your agent's output by adding a **Review Agent** that critiques and refines the results.

**Real-Life Scenario:**  
You are building a **Personal Nutrition Assistant** that helps users with meal planning and dietary advice.

The current agent generates meal suggestions, but the quality is inconsistent. Your task is to add a Review Agent that critiques the output and suggests improvements.

## 1. Setup

In [1]:
# Run this first
!pip install -q "langgraph" "langchain==0.3.*" "langchain-openai==0.2.*" "langchainhub" "langchain-community==0.3.*"  --force-reinstall
from datetime import datetime
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print(" Setup complete")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.4/155.4 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8

## 2. Base Tools

You can use these tools in your agents.

In [2]:
from langchain_core.tools import tool

@tool
def get_nutrition_info(food: str) -> str:
    """Get nutritional information for a food item."""
    data = {
        "tofu": "High in protein (8g per 100g), low in calories, good source of iron.",
        "quinoa": "Complete protein (14g per 100g), rich in fiber and minerals.",
        "greek yogurt": "High in protein (15-20g per serving), low in sugar if plain."
    }
    return data.get(food.lower(), f"Nutritional info for {food} is not available in this demo.")

## 3. Your Task

**Add a Review Agent to improve output quality.**

**Requirements:**

1. Create a **Generator Agent** that suggests meal ideas using the nutrition tool.
2. Create a **Review Agent** that critiques the generated meal plan and suggests improvements.
3. Implement a **review loop**: Generator → Reviewer → (optional revision).
4. Run a multi-turn test and show how the review improves the final output.

**Test Query:**  
"I am vegetarian and trying to lose weight. Suggest a high-protein breakfast and dinner plan."

## 4. Implement the Review Agent Workflow

Complete the code below:

In [4]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, Sequence
import operator
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

class AgentState(TypedDict):
    messages: Annotated[Sequence[str], operator.add]

def generator_node(state):
    """Generator Agent creates initial meal plan."""
    query = state["messages"][-1]
    prompt = f"Generate a vegetarian high-protein meal plan for: {query}"
    response = llm.invoke(prompt)
    return {"messages": state["messages"] + [f"Generator: {response.content}"]}

def reviewer_node(state):
    """Reviewer Agent critiques the previous output and suggests improvements."""
    last_output = state["messages"][-1]
    prompt = f"""Review the following meal plan and suggest improvements for accuracy, balance, and practicality:

{last_output}

Review:"""
    response = llm.invoke(prompt)
    return {"messages": state["messages"] + [f"Reviewer: {response.content}"]}

# Build the graph with review loop
workflow = StateGraph(AgentState)

workflow.add_node("generator", generator_node)
workflow.add_node("reviewer", reviewer_node)

workflow.set_entry_point("generator")
workflow.add_edge("generator", "reviewer")
workflow.add_edge("reviewer", END)   # Can be extended to revision loop

app = workflow.compile()

print(" Multi-Agent Workflow with Review Agent compiled successfully")

 Multi-Agent Workflow with Review Agent compiled successfully


## 5. Test Your Workflow

Run the following test query after completing the code:

In [5]:
test_query = "I am vegetarian and trying to lose weight. Suggest a high-protein breakfast and dinner plan."

initial_state = {"messages": [test_query]}

result = app.invoke(initial_state)

print("=== FINAL OUTPUT AFTER REVIEW ===")
print(result["messages"][-1])

=== FINAL OUTPUT AFTER REVIEW ===
Reviewer: Overall, this meal plan is quite balanced and includes a good variety of plant-based proteins, whole grains, and vegetables. However, there are a few suggestions for improvement:

1. Breakfast: 
- Consider adding a source of healthy fats, such as nuts or seeds, to the scrambled tofu for added satiety and nutrient balance.
- Ensure the whole grain toast is made from 100% whole grains for maximum nutritional benefit.

2. Dinner:
- While the lentil and vegetable stir-fry and chickpea curry are great plant-based protein options, consider adding a source of healthy fats, such as avocado or nuts, to the meals for added satiety.
- Ensure the quinoa salad is well-balanced with plenty of vegetables and a flavorful dressing for added taste.

3. Snacks:
- While hummus with carrot and cucumber sticks is a great snack option, consider adding a source of healthy fats, such as nuts or seeds, for added satiety.
- Ensure the protein smoothie is well-balanced 

## 6. Reflection

1. How did the Review Agent improve the quality of the meal suggestions?
2. Did the review loop make the output more balanced and useful?
3. What challenges did you face when implementing the reviewer?
4. In a real production system, how would you make the review process even better?